### Import all necessary libraries


In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_chroma import Chroma



C:\Users\HP\AppData\Local\Temp\ipykernel_14928\3324882322.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
e:\Achal Mate\GenAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

### Document Loader

In [3]:
loader = PyPDFLoader("AI_Application_Module.pdf")

documents = loader.load()

print(len(documents))
print(documents[0].page_content)
print(documents[0].metadata)

3
AI
 
Application
 
Module
 
Overview
 
The
 
AI
 
Application
 
Module
 
is
 
a
 
core
 
component
 
of
 
the
 
Applicant
 
Tracking
 
System
 
(ATS)
 
designed
 
to
 
automate
 
candidate
 
screening,
 
interview
 
evaluation,
 
and
 
hiring
 
recommendations.
 
By
 
leveraging
 
Artificial
 
Intelligence
 
(AI)
 
and
 
Natural
 
Language
 
Processing
 
(NLP),
 
this
 
module
 
improves
 
recruitment
 
efficiency,
 
reduces
 
manual
 
intervention,
 
and
 
ensures
 
data-driven
 
and
 
unbiased
 
hiring
 
decisions.
 
 
1
 
AI-Recommended
 
Jobs
 
Based
 
on
 
Candidate
 
Profile
 
This
 
feature
 
enables
 
intelligent
 
job
 
recommendations
 
for
 
candidates
 
based
 
on
 
their
 
profiles,
 
skills,
 
and
 
experience.
 
AI
 
Parsing
 
of
 
Candidate
 
Profile
 
The
 
system
 
analyzes
 
the
 
candidate’s
 
profile
 
and
 
resume
 
using
 
NLP
 
techniques
 
to
 
extract
 
key
 
information,
 
including:
 
●
 
Technical
 
and
 
functional
 
skills
 
●
 
Years
 
of
 
experience


### Chunking

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))
print(chunks[0].page_content)

Number of chunks: 12
AI
 
Application
 
Module
 
Overview
 
The
 
AI
 
Application
 
Module
 
is
 
a
 
core
 
component
 
of
 
the
 
Applicant
 
Tracking
 
System
 
(ATS)
 
designed
 
to
 
automate
 
candidate
 
screening,
 
interview
 
evaluation,
 
and
 
hiring
 
recommendations.
 
By
 
leveraging
 
Artificial
 
Intelligence
 
(AI)
 
and
 
Natural
 
Language
 
Processing
 
(NLP),
 
this
 
module
 
improves
 
recruitment
 
efficiency,
 
reduces
 
manual
 
intervention,
 
and
 
ensures
 
data-driven
 
and
 
unbiased


## Embedding model

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

display(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1188.21it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### Vectore Store

In [6]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
display(vectorstore)

### Retriver

In [7]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10
    }
)
display(retriever)

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001E4F5F73770>, search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 10})

In [ ]:
query = "What factors are considered while evaluating a candidate?"

retrieved_docs = retriever.invoke(query)
print("Len of docs",len(retrieved_docs))
for doc in retrieved_docs:
    print(doc.page_content)

In [8]:
llm1 = HuggingFaceEndpoint(
    repo_id="ibm-granite/granite-4.2-30b",
    task="text-generation",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    temperature=0,
)

model1 = ChatHuggingFace(llm=llm1)


In [ ]:
context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}
"""

answer_1 = model1.invoke(prompt)
print(answer_1.content)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.3
)

llm = HuggingFacePipeline(pipeline=pipe)
model = ChatHuggingFace(llm=llm)

In [ ]:
answer = model.invoke(prompt)
print(answer.content)

## Memory

In [9]:
# CONVERSATION BUFFER MEMORY

buffer_memory = []

question = "What factors are considered while evaluating a candidate?"
retrieved_docs = retriever.invoke(question)
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)
print(context)
prompt = f"""
You are an ATS AI assistant.


Relevant information from the PDF:
{context}

Current question:
{question}

Answer the current question using the previous conversation
and the PDF information.

Answer:
"""


result = model1.invoke(prompt)
print(result.content)
answer = result.content

buffer_memory.append({
    "question": question,
    "answer": answer
})

print(buffer_memory)

Score
 
and
 
Relevancy
 
Score,
 
the
 
system
 
provides
 
a
 
final
 
recommendation:
 
●
 
Strong
 
–
 
Highly
 
suitable
 
candidate
 
recommended
 
for
 
the
 
next
 
hiring
 
stage
 
●
 
Medium
 
–
 
Candidate
 
meets
 
most
 
requirements
 
and
 
may
 
proceed
 
with
 
further
 
evaluation
 
●
 
Weak
 
–
 
Candidate
 
does
 
not
 
sufficiently
 
meet
 
role
 
requirements

Score
 
and
 
Relevancy
 
Score,
 
the
 
system
 
provides
 
a
 
final
 
recommendation:
 
●
 
Strong
 
–
 
Highly
 
suitable
 
candidate
 
recommended
 
for
 
the
 
next
 
hiring
 
stage
 
●
 
Medium
 
–
 
Candidate
 
meets
 
most
 
requirements
 
and
 
may
 
proceed
 
with
 
further
 
evaluation
 
●
 
Weak
 
–
 
Candidate
 
does
 
not
 
sufficiently
 
meet
 
role
 
requirements

response
 
evaluation
 
and
 
ensure
 
consistency
 
in
 
scoring.
 
Candidate
 
Answer
 
Evaluation
 
Candidate
 
responses
 
are
 
evaluated
 
using
 
AI
 
and
 
NLP
 
techniques.
 
The
 
evaluation
 
considers:
 
●
 
Accuracy
 
a

In [10]:
question2 = "Explain the first factor in detail."

# Convert memory into text
history = ""

for item in buffer_memory:
    history += f"""
User: {item['question']}
Assistant: {item['answer']}
"""

prompt_to_retriver = f"""
You are an ATS AI assistant.

Previous conversation:
{history}

Current question:
{question2}

"""

# Retrieve documents
retrieved_docs = retriever.invoke(prompt_to_retriver)

context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

In [11]:
context

'response\n \nevaluation\n \nand\n \nensure\n \nconsistency\n \nin\n \nscoring.\n \nCandidate\n \nAnswer\n \nEvaluation\n \nCandidate\n \nresponses\n \nare\n \nevaluated\n \nusing\n \nAI\n \nand\n \nNLP\n \ntechniques.\n \nThe\n \nevaluation\n \nconsiders:\n \n●\n \nAccuracy\n \nand\n \ncorrectness\n \nof\n \nthe\n \nanswer\n \n●\n \nRelevance\n \nto\n \nthe\n \nasked\n \nquestion\n \n●\n \nDepth\n \nof\n \nsubject\n \nknowledge\n \n●\n \nClarity\n \nand\n \ncompleteness\n \nof\n \nexplanation\n \nAI\n \nInterview\n \nand\n \nRelevancy\n \nScore\n \nGeneration\n \nBased\n \non\n \nthe\n \nresponse\n\nresponse\n \nevaluation\n \nand\n \nensure\n \nconsistency\n \nin\n \nscoring.\n \nCandidate\n \nAnswer\n \nEvaluation\n \nCandidate\n \nresponses\n \nare\n \nevaluated\n \nusing\n \nAI\n \nand\n \nNLP\n \ntechniques.\n \nThe\n \nevaluation\n \nconsiders:\n \n●\n \nAccuracy\n \nand\n \ncorrectness\n \nof\n \nthe\n \nanswer\n \n●\n \nRelevance\n \nto\n \nthe\n \nasked\n \nquestion\n \n●\n \

In [12]:
prompt = f"""
You are an ATS AI assistant.

Previous conversation:
{history}

Relevant information from the PDF:
{context}

Current question:
{question2}

Answer the current question using the previous conversation
and the PDF information.

Answer:
"""

In [13]:
result = model1.invoke(prompt)
print(result.content)
answer2 = result.content


In [14]:
buffer_memory.append({
    "question": question2,
    "answer": answer2
})
print(buffer_memory)

[{'question': 'What factors are considered while evaluating a candidate?', 'answer': 'Based solely on the provided PDF information, the factors considered while evaluating a candidate\'s response are explicitly listed under the section **"Candidate Answer Evaluation"**. The evaluation focuses on the candidate\'s response itself, not the scoring tiers (Strong/Medium/Weak), which are final recommendations derived from the evaluation.\n\nThe specific factors considered are:\n\n*   **Accuracy and correctness of the answer**  \n*   **Relevance to the asked question**  \n*   **Depth of subject knowledge**  \n*   **Clarity and completeness of explanation**  \n\nThese four criteria are used by the AI and NLP techniques to assess candidate responses, as stated in the PDF: *"Candidate responses are evaluated using AI and NLP techniques. The evaluation considers: [bullet points above]."*  \n\nThe scoring tiers (Strong, Medium, Weak) and the final recommendation are outcomes of this evaluation pro

### Buffer Window Memory

In [16]:
# Retrieve documents
question = "How can this be used in the interview engine?"


In [17]:
# Convert memory into text
history = ""

for item in buffer_memory:
    history += f"""
User: {item['question']}
Assistant: {item['answer']}
"""

In [18]:

prompt = f"""
You are an ATS AI assistant.

Previous conversation:
{history}


Current question:
{question}

Answer the current question using the previous conversation
and the PDF information.

Answer:
"""
retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)
context

'2\n \nAI\n \nInterview\n \nEngine\n \nand\n \nEvaluation\n \nOutput\n \nThe\n \nAI\n \nInterview\n \nEngine\n \nautomates\n \nthe\n \ninterview\n \nprocess\n \nby\n \ngenerating\n \nquestions,\n \nevaluating\n \nresponses,\n \nscoring\n \ncandidates,\n \nand\n \nproducing\n \nfinal\n \nhiring\n \nrecommendations.\n \nJD-Based\n \nQuestion\n \nGeneration\n \nBased\n \non\n \nthe\n \nparsed\n \nJob\n \nDescription,\n \nthe\n \nAI\n \ndynamically\n \ngenerates\n \nfive\n \nrole-specific\n \ninterview\n \nquestions.\n \nThese\n \nquestions:\n \n●\n \nAre\n \naligned\n \nwith\n \nthe\n \nrequired\n\n2\n \nAI\n \nInterview\n \nEngine\n \nand\n \nEvaluation\n \nOutput\n \nThe\n \nAI\n \nInterview\n \nEngine\n \nautomates\n \nthe\n \ninterview\n \nprocess\n \nby\n \ngenerating\n \nquestions,\n \nevaluating\n \nresponses,\n \nscoring\n \ncandidates,\n \nand\n \nproducing\n \nfinal\n \nhiring\n \nrecommendations.\n \nJD-Based\n \nQuestion\n \nGeneration\n \nBased\n \non\n \nthe\n \nparsed\n \nJ

In [19]:

prompt = f"""
You are an ATS AI assistant.

Previous conversation:
{history}

Relevant information from the PDF:
{context}

Current question:
{question}

Answer the current question using the previous conversation
and the PDF information.

Answer:
"""

In [20]:

result = model1.invoke(question)
answer = result.content

buffer_memory.append({
    "question": question,
    "answer": answer
})


In [21]:
buffer_window_memory = buffer_memory.copy()
print(len(buffer_window_memory))
# Number of conversations to remember
k = 2

3


In [22]:
buffer_window_memory = buffer_window_memory[-k:]

print(len(buffer_window_memory))

2


In [24]:
question = "Explain scroing process"
retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)
# Convert memory into text
history = ""

for item in buffer_window_memory:
    history += f"""
User: {item['question']}
Assistant: {item['answer']}
"""


prompt = f"""
You are an ATS AI assistant.

Previous conversation:
{history}

Relevant information from the PDF:
{context}

Current question:
{question}

Answer the current question using the previous conversation
and the PDF information.

Answer:
"""

result = model1.invoke(question)



In [25]:
# Convert memory into text
history = ""

for item in buffer_memory:
    history += f"""
User: {item['question']}
Assistant: {item['answer']}
"""

prompt = f"""
Create the summary of {history}
"""

summary = model1.invoke(prompt)


In [ ]:

prompt = f"""
Update the conversation summary.
Keep important facts, user preferences, decisions and references needed for future questions.
Do not invent information.

Previous summary:
{summary}

New user message:
{question1}

Updated summary:
"""